# ASVspoof5 Cross-Gender Transfer Evaluation (A01-A08)

This notebook reuses already-trained gender-specific logistic models and evaluates transfer:
- `male -> female`
- `female -> male`

For each task (`bonafide` vs `A01..A08`), it evaluates on the **target gender test split**.


In [ ]:
import io
import gc
import json
import pickle
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchaudio
from tqdm import tqdm

from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report, roc_curve

import matplotlib.pyplot as plt
from IPython.display import display


In [ ]:
# ===== Config =====
PROJECT_ROOT = Path('/home/SpeakerRec/BioVoice')

PLAN_BASE = (
    PROJECT_ROOT
    / "data"
    / "datasets"
    / "ASVspoof5_tars"
    / "ASVspoof5_protocols"
    / "gender_50spk_2000perclass_A01A08_outputs"
)
MANIFESTS = {
    'male': PLAN_BASE / 'male' / 'selected_utterances_plan.csv',
    'female': PLAN_BASE / 'female' / 'selected_utterances_plan.csv',
}

TRAIN_TAR_DIR = PROJECT_ROOT / 'data' / 'datasets' / 'ASVspoof5_tars' / 'ASVspoof5_audio_train_tars'
TRAINED_MODELS_BASE = PROJECT_ROOT / 'data' / 'models' / 'asvspoof5_train_gender_50spk_2000perclass_A01A08'
OUT_BASE = PROJECT_ROOT / 'data' / 'models' / 'asvspoof5_train_gender_50spk_2000perclass_A01A08_cross_gender_eval'
OUT_BASE.mkdir(parents=True, exist_ok=True)

TARGET_SYSTEMS = ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08']
TRANSFER_PAIRS = [('male', 'female'), ('female', 'male')]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FORCE_RECOMPUTE_EMBEDDINGS = False
SAVE_PREDICTIONS = True

print('DEVICE =', DEVICE)
print('TRAIN_TAR_DIR =', TRAIN_TAR_DIR, '| exists =', TRAIN_TAR_DIR.exists())
print('TRAINED_MODELS_BASE =', TRAINED_MODELS_BASE, '| exists =', TRAINED_MODELS_BASE.exists())
print('OUT_BASE =', OUT_BASE)
for g, m in MANIFESTS.items():
    print(f'{g} manifest = {m} | exists =', m.exists())


In [ ]:
# ===== Load ReDimNet =====
redim_model = (
    torch.hub.load(
        'IDRnD/ReDimNet',
        'ReDimNet',
        model_name='b6',
        train_type='ptn',
        dataset='vox2',
    )
    .to(DEVICE)
    .eval()
)
print('Loaded ReDimNet on', DEVICE)


In [ ]:
def embed_waveform(wav: torch.Tensor, sr: int) -> np.ndarray:
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    if wav.ndim == 1:
        wav = wav.unsqueeze(0)
    if wav.shape[0] > 1:
        wav = wav[:1, :]
    wav = wav.to(DEVICE)
    with torch.no_grad():
        emb = redim_model(wav)
    return emb.squeeze(0).detach().cpu().numpy().astype(np.float32)


def embed_member_from_tar(tf: tarfile.TarFile, member: tarfile.TarInfo) -> np.ndarray:
    fobj = tf.extractfile(member)
    if fobj is None:
        raise RuntimeError(f'Cannot extract member: {member.name}')
    raw = fobj.read()
    try:
        wav, sr = torchaudio.load(io.BytesIO(raw))
    except Exception:
        import tempfile
        suffix = Path(member.name).suffix or '.flac'
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=True) as tmp:
            tmp.write(raw)
            tmp.flush()
            wav, sr = torchaudio.load(tmp.name)
    return embed_waveform(wav, sr)


def extract_embeddings_for_manifest(manifest_df: pd.DataFrame, cache_npz: Path, force_recompute: bool = False):
    if cache_npz.exists() and not force_recompute:
        payload = np.load(cache_npz, allow_pickle=True)
        X = payload['X']
        utt_ids = payload['utt_ids'].astype(str)
        lut = pd.DataFrame({'utt_id': utt_ids, '_idx': np.arange(len(utt_ids))})
        m = manifest_df[['utt_id']].merge(lut, on='utt_id', how='left', validate='one_to_one')
        if m['_idx'].isna().any():
            miss = m.loc[m['_idx'].isna(), 'utt_id'].tolist()[:10]
            raise RuntimeError(f'Embedding cache missing utt_ids, examples={miss}')
        return X[m['_idx'].astype(int).to_numpy()]

    tars = sorted(TRAIN_TAR_DIR.glob('flac_T_*.tar'))
    assert len(tars) > 0, f'No train tar files in {TRAIN_TAR_DIR}'
    need = set(manifest_df['utt_id'].astype(str).tolist())
    emb_map = {}
    found = set()

    for tar_path in tars:
        print('Reading', tar_path.name)
        with tarfile.open(tar_path, 'r') as tf:
            for m in tf:
                if not m.isfile():
                    continue
                utt = Path(Path(m.name).name).stem
                if utt not in need or utt in found:
                    continue
                emb_map[utt] = embed_member_from_tar(tf, m)
                found.add(utt)
        print('Found so far:', len(found), '/', len(need))

    missing = sorted(list(need - found))
    if missing:
        raise RuntimeError(f'Missing {len(missing)} utt_ids in train tars. examples={missing[:10]}')

    ids = manifest_df['utt_id'].astype(str).tolist()
    X = np.stack([emb_map[u] for u in ids]).astype(np.float32)
    np.savez_compressed(cache_npz, X=X, utt_ids=np.array(ids, dtype=object))
    return X


def compute_metrics(y_true, p_spoof, thr=0.5):
    y_hat = (p_spoof >= thr).astype(int)
    cm = confusion_matrix(y_true, y_hat).tolist()
    out = {
        'accuracy': float(accuracy_score(y_true, y_hat)),
        'auc': float(roc_auc_score(y_true, p_spoof)) if len(np.unique(y_true)) == 2 else None,
        'confusion_matrix': cm,
        'classification_report': classification_report(y_true, y_hat, output_dict=True, zero_division=0),
    }
    try:
        fpr, tpr, thr_vals = roc_curve(y_true, p_spoof)
        out['roc'] = {
            'fpr': fpr.tolist(),
            'tpr': tpr.tolist(),
            'thresholds': thr_vals.tolist(),
        }
    except Exception:
        out['roc'] = None
    return out


In [ ]:
# ===== Load manifests and build embedding caches (target test split only) =====
manifest_by_gender = {}
test_manifest_by_gender = {}
X_test_by_gender = {}

for gender_name, manifest_path in MANIFESTS.items():
    df = pd.read_csv(manifest_path)
    req_cols = {'split', 'speaker_id', 'utt_id', 'gender', 'target_class'}
    missing = req_cols - set(df.columns)
    if missing:
        raise ValueError(f'{gender_name} manifest missing cols: {sorted(missing)}')

    df = df[df['target_class'].isin(['bonafide'] + TARGET_SYSTEMS)].copy().reset_index(drop=True)
    df_test = df[df['split'].eq('test')].copy().reset_index(drop=True)
    if df_test.empty:
        raise RuntimeError(f'{gender_name}: test split is empty')

    g_dir = OUT_BASE / '_embedding_cache' / gender_name
    g_dir.mkdir(parents=True, exist_ok=True)
    cache_npz = g_dir / 'embeddings_test_only.npz'
    X_test = extract_embeddings_for_manifest(df_test[['utt_id']], cache_npz, force_recompute=FORCE_RECOMPUTE_EMBEDDINGS)

    manifest_by_gender[gender_name] = df
    test_manifest_by_gender[gender_name] = df_test
    X_test_by_gender[gender_name] = X_test

    print(f'{gender_name}: test rows =', len(df_test), '| cache =', cache_npz)


In [ ]:
# ===== Cross-gender evaluation =====
rows = []

for src_gender, tgt_gender in TRANSFER_PAIRS:
    print(f'\n===== Transfer {src_gender} -> {tgt_gender} =====')
    pair_dir = OUT_BASE / f'{src_gender}_to_{tgt_gender}'
    pair_dir.mkdir(parents=True, exist_ok=True)

    tgt_df = test_manifest_by_gender[tgt_gender]
    X_tgt_all = X_test_by_gender[tgt_gender]
    idx_map = pd.DataFrame({'utt_id': tgt_df['utt_id'].astype(str), '_idx': np.arange(len(tgt_df))})

    for sys_id in TARGET_SYSTEMS:
        print(f'-- Task bonafide vs {sys_id}')
        model_dir = TRAINED_MODELS_BASE / src_gender / sys_id
        scaler_path = model_dir / 'scaler.pkl'
        clf_path = model_dir / 'logistic_regression.pkl'
        if not scaler_path.exists() or not clf_path.exists():
            raise FileNotFoundError(f'Missing model artifacts in {model_dir}')

        with open(scaler_path, 'rb') as f:
            scaler = pickle.load(f)
        with open(clf_path, 'rb') as f:
            clf = pickle.load(f)

        task_df = tgt_df[tgt_df['target_class'].isin(['bonafide', sys_id])].copy().reset_index(drop=True)
        if task_df.empty:
            raise RuntimeError(f'Empty target task data for {tgt_gender}/{sys_id}')

        pick = task_df[['utt_id']].astype(str).merge(idx_map, on='utt_id', how='left', validate='one_to_one')
        if pick['_idx'].isna().any():
            raise RuntimeError(f'Index mapping failed for {tgt_gender}/{sys_id}')

        X_te = X_tgt_all[pick['_idx'].astype(int).to_numpy()]
        y_te = np.where(task_df['target_class'].eq('bonafide'), 0, 1).astype(int)

        if set(np.unique(y_te)) != {0, 1}:
            raise RuntimeError(f'Target class imbalance for {tgt_gender}/{sys_id}')

        X_te_s = scaler.transform(X_te)
        p_te = clf.predict_proba(X_te_s)[:, 1]
        m_te = compute_metrics(y_te, p_te, thr=0.5)

        task_dir = pair_dir / sys_id
        task_dir.mkdir(parents=True, exist_ok=True)

        run_summary = {
            'source_gender_model': src_gender,
            'target_gender_data': tgt_gender,
            'task': f'bonafide_vs_{sys_id}',
            'test_rows': int(len(task_df)),
            'metrics_test_thr_0_5': m_te,
        }
        (task_dir / 'run_summary.json').write_text(json.dumps(run_summary, indent=2), encoding='utf-8')

        if SAVE_PREDICTIONS:
            pred = task_df[['split', 'speaker_id', 'utt_id', 'gender', 'target_class']].copy().reset_index(drop=True)
            pred['label_id'] = y_te
            pred['prob_spoof'] = p_te
            pred['pred_label_id_thr_0_5'] = (p_te >= 0.5).astype(int)
            pred.to_csv(task_dir / f'predictions_transfer_{src_gender}_to_{tgt_gender}_bonafide_vs_{sys_id}.csv', index=False)

        if m_te['roc'] is not None:
            roc_df = pd.DataFrame({
                'fpr': m_te['roc']['fpr'],
                'tpr': m_te['roc']['tpr'],
                'threshold': m_te['roc']['thresholds'],
            })
            roc_df.to_csv(task_dir / 'roc_curve_points.csv', index=False)

        rows.append({
            'source_gender_model': src_gender,
            'target_gender_data': tgt_gender,
            'task_system_id': sys_id,
            'test_n': int(len(y_te)),
            'test_acc': m_te['accuracy'],
            'test_auc': m_te['auc'],
            'test_cm_00': int(m_te['confusion_matrix'][0][0]),
            'test_cm_01': int(m_te['confusion_matrix'][0][1]),
            'test_cm_10': int(m_te['confusion_matrix'][1][0]),
            'test_cm_11': int(m_te['confusion_matrix'][1][1]),
        })

        del X_te, y_te, X_te_s, p_te
        gc.collect()

summary_df = pd.DataFrame(rows).sort_values(['source_gender_model', 'target_gender_data', 'task_system_id']).reset_index(drop=True)
summary_csv = OUT_BASE / 'cross_gender_metrics_summary.csv'
summary_df.to_csv(summary_csv, index=False)

print('Saved:', summary_csv)
display(summary_df)


In [ ]:
# ===== Quick plots: AUC and confusion matrix bars =====
summary_csv = OUT_BASE / 'cross_gender_metrics_summary.csv'
df = pd.read_csv(summary_csv)

display(df)

for pair_name, g in df.groupby(['source_gender_model', 'target_gender_data']):
    src, tgt = pair_name
    title = f'{src} -> {tgt}'

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    g2 = g.sort_values('task_system_id')
    axes[0].bar(g2['task_system_id'], g2['test_auc'])
    axes[0].set_title(f'Test AUC ({title})')
    axes[0].set_ylim(0.0, 1.0)
    axes[0].set_ylabel('AUC')

    cm_err = (g2['test_cm_01'] + g2['test_cm_10']).to_numpy()
    axes[1].bar(g2['task_system_id'], cm_err)
    axes[1].set_title(f'Total test errors ({title})')
    axes[1].set_ylabel('Count')

    plt.tight_layout()
    plt.show()
